# Vision Prompt Eval — Garmin Activity Screens

Runs the current `ingestion/health.py` prompt against the labeled screenshots in `data/drop/processed/` and scores the output field-by-field against the hand-verified ground truth in `eval_data/vision_labels.json`.

Re-run this notebook after any prompt/schema change in `ingestion/health.py` to check whether accuracy actually improved — don't trust a single run, the vision model is not fully deterministic (see `eval_data/vision_labels.json`'s `_year_inference_rule` note and prior findings on `duration_seconds` noise).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

import json
from datetime import date

from config.settings import DROP_FOLDER
from ingestion.health import PROMPT, _parse_metrics, _query_ollama, _resolve_date

LABELS_PATH = Path("eval_data") / "vision_labels.json"
PROCESSED_DIR = DROP_FOLDER / "processed"

print(PROMPT)

You are extracting data from a Garmin fitness tracker screenshot.

First, decide which type of screen this is:
- ACTIVITY screen: shows a specific workout (e.g. a run, walk, or ride) with metrics like distance, duration, pace, or workout heart rate. There may be multiple activities listed.
- DAILY SUMMARY screen: shows aggregate stats for the whole day — total steps, active minutes, resting heart rate. No distance or pace is shown.

Only extract a value if that specific activity's own row shows a labeled field for it. Never copy or infer a value from a different metric, or from a different activity's row — if a field isn't present for this activity, return null for it.

Screenshots rarely show a year — do not guess one. Only extract the month and day shown.

If this is an ACTIVITY screen, return a JSON array with one object per activity — even if there is only one:
[
  {
    "type": "activity",
    "month_day": "MM-DD",
    "activity_type": <"running", "walking", "cycling", "strength t

## 1. Load the labeled test set

In [2]:
labels = json.loads(LABELS_PATH.read_text())
print(f"{len(labels['images'])} labeled images")
for img in labels["images"]:
    print(f"  {img['file']} — {len(img['expected'])} activities, year_visible_in_image={img['year_visible_in_image']}")

4 labeled images
  Screenshot 2026-05-10 at 11.42.06 AM.png — 1 activities, year_visible_in_image=True
  Screenshot 2026-05-11 at 8.41.22 PM.png — 3 activities, year_visible_in_image=True
  Screenshot 2026-05-12 at 8.07.07 PM.png — 2 activities, year_visible_in_image=True
  Screenshot 2026-07-09 at 19.10.28.png — 1 activities, year_visible_in_image=False


## 2. Scoring helpers

Compares each field of the model's parsed output against the `expected` record for that activity. Numeric fields get a small float tolerance; everything else is exact match. `None` only matches `None`.

In [3]:
FIELDS = [
    "type",
    "date",
    "activity_type",
    "distance_miles",
    "duration_seconds",
    "avg_pace_seconds_per_mile",
    "avg_hr",
    "max_hr",
    "calories",
]
NUMERIC_FIELDS = {
    "distance_miles",
    "duration_seconds",
    "avg_pace_seconds_per_mile",
    "avg_hr",
    "max_hr",
    "calories",
}


def values_match(field, expected, actual):
    if expected is None or actual is None:
        return expected == actual
    if field in NUMERIC_FIELDS:
        try:
            return abs(float(expected) - float(actual)) < 0.01
        except (TypeError, ValueError):
            return expected == actual
    return expected == actual


def score_record(expected, actual):
    return {field: values_match(field, expected.get(field), actual.get(field)) for field in FIELDS}

## 3. Run every image through the model and score

Uses the real `_query_ollama` / `_parse_metrics` / `_resolve_date` from `ingestion/health.py`, so this tests the actual ingestion code path, not a reimplementation of it.

In [4]:
field_hits = {f: 0 for f in FIELDS}
field_total = {f: 0 for f in FIELDS}
perfect_records = 0
total_records = 0
count_mismatches = []

for image in labels["images"]:
    image_path = PROCESSED_DIR / image["file"]
    print(f"\n=== {image['file']} ===")
    if not image_path.exists():
        print(f"  MISSING FILE at {image_path} — skipping")
        continue

    raw = _query_ollama(image_path)
    records = _parse_metrics(raw) or []
    for record in records:
        record["date"] = _resolve_date(record.get("month_day"), date.today())

    expected_records = image["expected"]

    if len(records) != len(expected_records):
        print(f"  ACTIVITY COUNT MISMATCH: expected {len(expected_records)}, got {len(records)}")
        count_mismatches.append(image["file"])

    # Positional matching — assumes cards come back top-to-bottom same as expected.
    # A count mismatch above means this comparison is already misaligned; treat
    # per-field results for that image as informational, not authoritative.
    for i, expected in enumerate(expected_records):
        total_records += 1
        actual = records[i] if i < len(records) else {}
        scored = score_record(expected, actual)
        if all(scored.values()):
            perfect_records += 1

        for field, ok in scored.items():
            field_total[field] += 1
            if ok:
                field_hits[field] += 1
            else:
                print(f"  [activity {i}] {field}: expected={expected.get(field)!r} actual={actual.get(field)!r}")


=== Screenshot 2026-05-10 at 11.42.06 AM.png ===



=== Screenshot 2026-05-11 at 8.41.22 PM.png ===


  [activity 0] duration_seconds: expected=1220 actual=1240

=== Screenshot 2026-05-12 at 8.07.07 PM.png ===



=== Screenshot 2026-07-09 at 19.10.28.png ===


## 4. Field accuracy summary

In [5]:
print("FIELD ACCURACY")
for field in FIELDS:
    total = field_total[field]
    hits = field_hits[field]
    pct = (hits / total * 100) if total else 0.0
    print(f"  {field:28s} {hits:2d}/{total:2d}  ({pct:5.1f}%)")

print(f"\nPERFECT RECORDS: {perfect_records}/{total_records}")
if count_mismatches:
    print(f"ACTIVITY COUNT MISMATCHES: {count_mismatches}")

FIELD ACCURACY
  type                          7/ 7  (100.0%)
  date                          7/ 7  (100.0%)
  activity_type                 7/ 7  (100.0%)
  distance_miles                7/ 7  (100.0%)
  duration_seconds              6/ 7  ( 85.7%)
  avg_pace_seconds_per_mile     7/ 7  (100.0%)
  avg_hr                        7/ 7  (100.0%)
  max_hr                        7/ 7  (100.0%)
  calories                      7/ 7  (100.0%)

PERFECT RECORDS: 6/7
